In [1]:
"Can we put a VWR 100ul V-bottom plate on a PLT_CAR_L5PCR_A01 carrier?"



'Can we put a VWR 100ul V-bottom plate on a PLT_CAR_L5PCR_A01 carrier?'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, TIP_CAR_480_A00
from pylabrobot.resources.hamilton import PLT_CAR_L5MD_A00
from pylabrobot.resources.hamilton import Hamilton_96_adapter_188182

from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL_filter,     # 1000 µL filtered 
    hamilton_96_tiprack_10uL_filter #Tip Rack with 96 10ul Low Volume Tip with filter
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

2026-02-25 12:21:49,351 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-25 12:21:49,359 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-25 12:21:49,363 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-25 12:21:52,547 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

def VWR_96_wellplate_100_Vb_on_starCarrier_1820701(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
It is currently on a STARLet carrier with catalog number 182070, and the plate is modeled on the carrier.
  """
  
  return Plate(
    name=name,
    size_x=127.55,
    size_y=85.0,
    size_z=18.6,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb_on_starCarrier_1820701.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11,  # measured
      dy=10,  # measured
      dz=2.15, # measured
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=16, # measured well depth
      material_z_thickness=1.0,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

In [5]:
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

# Tip racks on rails=25, module slots 0,1,2,4
tiprack_1000 = hamilton_96_tiprack_1000uL_filter("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_pcr = hamilton_96_tiprack_50uL_filter("tips_pcr")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10
tip_car[4] = tiprack_pcr

In [8]:



# PCR plates on rail 1 all modules, mastermix plate on module 0

#make a plate carrier
plate_carrier = PLT_CAR_L5MD_A00("plate_carrier")
lh.deck.assign_child_resource(plate_carrier, rails=1) # assign the plate carrior to the deck
#make a plate adapter
pcr_mastermix_plate_adapter = Hamilton_96_adapter_188182("pcr_mastermix_plate_adapter")
#make a PCR plate
pcr_mastermix_plate = VWR_96_wellplate_100_Vb_on_starCarrier_1820701("pcr_mastermix_plate")

#assign the plate carrier position[0] to the plate adapter
plate_carrier[0].assign_child_resource(pcr_mastermix_plate_adapter)
#assign the PCR plate to the adapter
pcr_mastermix_plate_adapter.assign_child_resource(pcr_mastermix_plate)
# car_1 = PLT_CAR_L5MD_A00("car_1", modules={0:pcr_mastermix_module})






In [11]:
CHANNELS_8=[0,1,2,3,4,5,6,7]

# await lh.pick_up_tips(tiprack_1000["A1:H1"], use_channels=CHANNELS_8)  
await lh.aspirate(
    pcr_mastermix_plate["A2:H2"],
    vols=[100]*8,
    use_channels=CHANNELS_8,
    liquid_height=[8]*8,    
)

await lh.dispense(
    pcr_mastermix_plate["A2:H2"],
    vols=[30]*8,
    use_channels=CHANNELS_8,
    liquid_height=[8]*8,
    settling_time=[1]*8
    
)